# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sukhman-builds/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [21]:
!git clone --depth 1 https://github.com/flyrank-bih/flyrank-ml-internship-starter.git /content/flyrank-starter

!mkdir -p /content/data/raw

!cp /content/flyrank-starter/data/raw/content_refresh_anonymized.csv /content/data/raw/

import os
os.chdir("/content")

print("CSV ready:", os.path.exists("data/raw/content_refresh_anonymized.csv"))

fatal: destination path '/content/flyrank-starter' already exists and is not an empty directory.
CSV ready: True


## 1. My Rule and Signal Checks

### Lane
Lane 2 — Content Refresh & Opportunity Scoring.

### Heuristic Rule

Prioritize pages that meet all three conditions:

1. Not updated for at least 180 days.
2. At least 500 organic impressions in the last 90 days.
3. Average search position between 1 and 10.

This is a decision-support heuristic, not proof that refreshing a page will improve its performance.

### Signal 1 — Content Staleness

Feature: `days_since_last_update`

Hypothesis: Older content may need editorial review.

Observed bucket-level decline rates:

| Bucket | Pages (n) | Decline rate |
|---|---:|---:|
| 0–30 | 20,480 | 51.14% |
| 31–90 | 175 | 58.86% |
| 91–180 | 9,171 | 61.11% |
| 181–365 | 169 | 46.75% |
| 365+ | 5 | 60.00% |

Verdict: MIXED

The decline rate does not increase consistently with content age. The 365+ bucket has only 5 pages, so its result is unreliable.

### Signal 2 — Organic Search Visibility

Feature: `impressions_90d`

Hypothesis: Pages with more impressions may offer a larger observable search opportunity.

Observed bucket-level decline rates:

| Bucket | Pages (n) | Decline rate |
|---|---:|---:|
| Below 100 | 7,994 | 38.90% |
| 100–499 | 5,280 | 60.44% |
| 500–2,999 | 8,443 | 62.06% |
| 3,000+ | 8,283 | 57.00% |

Verdict: MIXED

Decline rates rise across the first three buckets but decrease in the highest visibility bucket.

### Overall Finding

Dataset size: 30,000 rows and 44 columns.

Overall observed decline rate: 54.21%.

Neither signal shows a consistent pattern across all buckets. These observations are directional and do not establish causation.

### Reason Code

`STALE_VISIBLE_FIRST_PAGE`

Assigned when all three priority conditions are met.

In [22]:
import pandas as pd
import numpy as np
from pathlib import Path

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Dataset shape:", df.shape)

required = [
    "content_id",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "trend_direction"
]

missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

df["audit_label"] = (
    df["trend_direction"].astype(str).str.lower().eq("down").astype(int)
)

df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[-np.inf, 30, 90, 180, 365, np.inf],
    labels=["0-30", "31-90", "91-180", "181-365", "365+"]
)

staleness_table = (
    df.groupby("staleness_bucket", observed=False)
    .agg(n=("content_id", "size"), decline_rate=("audit_label", "mean"))
    .reset_index()
)

staleness_table["decline_rate_pct"] = (
    staleness_table["decline_rate"] * 100
).round(2)

print("SIGNAL 1 — STALENESS")
display(staleness_table)

df["visibility_bucket"] = pd.cut(
    df["impressions_90d"],
    bins=[-np.inf, 99, 499, 2999, np.inf],
    labels=["Below 100", "100-499", "500-2999", "3000+"]
)

visibility_table = (
    df.groupby("visibility_bucket", observed=False)
    .agg(n=("content_id", "size"), decline_rate=("audit_label", "mean"))
    .reset_index()
)

visibility_table["decline_rate_pct"] = (
    visibility_table["decline_rate"] * 100
).round(2)

print("SIGNAL 2 — ORGANIC VISIBILITY")
display(visibility_table)

Dataset shape: (30000, 44)
SIGNAL 1 — STALENESS


,staleness_bucket,n,decline_rate,decline_rate_pct
0,0-30,20480,0.511377,51.14
1,31-90,175,0.588571,58.86
2,91-180,9171,0.611057,61.11
3,181-365,169,0.467456,46.75
4,365+,5,0.600000,60.00


SIGNAL 2 — ORGANIC VISIBILITY


,visibility_bucket,n,decline_rate,decline_rate_pct
0,Below 100,7994,0.389042,38.90
1,100-499,5280,0.604356,60.44
2,500-2999,8443,0.620632,62.06
3,3000+,8283,0.569963,57.00


## 2. Build the ranked queue (writes the CSV)

## 2. Build the Ranked Queue

### Baseline Scoring Rule

The baseline score uses three observable features:

| Condition | Points |
|---|---:|
| At least 180 days since last update | 40 |
| At least 500 impressions in 90 days | 35 |
| Average position between 1 and 10 | 25 |

Maximum possible score: 100.

### Action Labels

- `REFRESH_PRIORITY`: All three conditions are met.
- `MONITOR_REVIEW`: Score is at least 40 but all priority conditions are not met.
- `NO_ACTION`: Score is below 40.

### Reason Codes

- `STALE_VISIBLE_FIRST_PAGE`: All three priority conditions are met.
- `CRITERIA_NOT_MET`: The complete priority rule is not met.

### Observed Result

- Total pages ranked: 30,000
- REFRESH_PRIORITY: 3
- MONITOR_REVIEW: 7,732
- NO_ACTION: 22,265

The ranked queue is sorted by baseline score and then by impressions.

Output: `work/outputs/baseline_action_score.csv`

The scoring formula uses observable input features only. It does not use the decline label as a scoring feature.

In [23]:
# 1. Score each condition

df["stale_points"] = (
    df["days_since_last_update"] >= 180
).astype(int) * 40

df["visibility_points"] = (
    df["impressions_90d"] >= 500
).astype(int) * 35

df["first_page_points"] = (
    (df["avg_position"] > 0) &
    (df["avg_position"] <= 10)
).astype(int) * 25

# 2. Calculate final score

df["baseline_score"] = (
    df["stale_points"] +
    df["visibility_points"] +
    df["first_page_points"]
)

# 3. Define priority condition

priority_condition = (
    (df["days_since_last_update"] >= 180) &
    (df["impressions_90d"] >= 500) &
    (df["avg_position"] > 0) &
    (df["avg_position"] <= 10)
)

# 4. Assign action and reason code

df["reason_code"] = np.where(
    priority_condition,
    "STALE_VISIBLE_FIRST_PAGE",
    "CRITERIA_NOT_MET"
)

df["action_label"] = np.where(
    priority_condition,
    "REFRESH_PRIORITY",
    np.where(
        df["baseline_score"] >= 40,
        "MONITOR_REVIEW",
        "NO_ACTION"
    )
)

# 5. Rank pages

df_ranked = df.sort_values(
    ["baseline_score", "impressions_90d"],
    ascending=[False, False]
).reset_index(drop=True)

df_ranked.insert(0, "rank", range(1, len(df_ranked) + 1))

# 6. Save ranked queue

output_dir = Path("work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

output_columns = [
    "rank",
    "content_id",
    "baseline_score",
    "action_label",
    "reason_code",
    "days_since_last_update",
    "impressions_90d",
    "avg_position"
]

csv_path = output_dir / "baseline_action_score.csv"

df_ranked[output_columns].to_csv(csv_path, index=False)

# 7. Show results

print("Ranked queue generated:", len(df_ranked), "rows")
print("Saved to:", csv_path)

print("\nTOP 5 PAGES")
display(df_ranked[output_columns].head(5))

print("\nACTION SUMMARY")
display(df_ranked["action_label"].value_counts().to_frame("count"))

Ranked queue generated: 30000 rows
Saved to: work/outputs/baseline_action_score.csv

TOP 5 PAGES


,rank,content_id,baseline_score,action_label,reason_code,days_since_last_update,impressions_90d,avg_position
0,1,content_e3ff1b093148,100,REFRESH_PRIORITY,STALE_VISIBLE_FIRST_PAGE,183,1408,7.8
1,2,content_7f116ae1f6f5,100,REFRESH_PRIORITY,STALE_VISIBLE_FIRST_PAGE,301,954,9.0
2,3,content_72496874f806,100,REFRESH_PRIORITY,STALE_VISIBLE_FIRST_PAGE,301,821,5.8
3,4,content_cf56e2e2e282,75,MONITOR_REVIEW,CRITERIA_NOT_MET,194,61678,19.7
4,5,content_7368877ea310,75,MONITOR_REVIEW,CRITERIA_NOT_MET,194,59472,24.8



ACTION SUMMARY


,count
action_label,
NO_ACTION,22265
MONITOR_REVIEW,7732
REFRESH_PRIORITY,3


## 3. Top-20 review

### Top-20 Manual Review

## 3. Top-20 Manual Review

The 20 highest-ranked pages are reviewed individually.

Each review includes:

- Rank and anonymized content ID.
- Baseline score and assigned action.
- Reason code.
- Confidence note based on the number of rule conditions met.
- A possible failure mode that could make the recommendation incorrect.

### Confidence Interpretation

- High: 3 out of 3 conditions met.
- Medium: 2 out of 3 conditions met.
- Low: 0 or 1 condition met.

These confidence notes describe rule evidence only. They are not calibrated probabilities or guarantees of success.

### Review Limitation

A page can satisfy the scoring conditions and still be a poor refresh candidate because of seasonality, technical problems, or changes in search results.

In [24]:
top20 = df_ranked.head(20).copy()

def confidence_note(row):
    checks = [
        row["days_since_last_update"] >= 180,
        row["impressions_90d"] >= 500,
        0 < row["avg_position"] <= 10
    ]

    count = sum(checks)

    if count == 3:
        return "High: 3/3 conditions met"
    elif count == 2:
        return "Medium: 2/3 conditions met"
    else:
        return f"Low: {count}/3 conditions met"


def failure_mode(row):
    if row["impressions_90d"] < 500:
        return "Low search volume may limit opportunity"
    elif row["avg_position"] <= 0:
        return "Position data is unavailable"
    elif row["avg_position"] > 10:
        return "Page is outside first-page ranking"
    elif row["days_since_last_update"] < 180:
        return "Page was updated recently"
    else:
        return "Seasonality or technical issues may explain decline"


top20["confidence_note"] = top20.apply(confidence_note, axis=1)
top20["what_would_make_it_wrong"] = top20.apply(failure_mode, axis=1)

review_columns = [
    "rank",
    "content_id",
    "baseline_score",
    "action_label",
    "reason_code",
    "confidence_note",
    "what_would_make_it_wrong"
]

print("TOP-20 MANUAL REVIEW")
display(top20[review_columns])

assert len(top20) == 20

print("Top-20 review completed:", len(top20), "pages")

TOP-20 MANUAL REVIEW


,rank,content_id,baseline_score,action_label,reason_code,confidence_note,what_would_make_it_wrong
0,1,content_e3ff1b093148,100,REFRESH_PRIORITY,STALE_VISIBLE_FIRST_PAGE,High: 3/3 conditions met,Seasonality or technical issues may explain de...
1,2,content_7f116ae1f6f5,100,REFRESH_PRIORITY,STALE_VISIBLE_FIRST_PAGE,High: 3/3 conditions met,Seasonality or technical issues may explain de...
2,3,content_72496874f806,100,REFRESH_PRIORITY,STALE_VISIBLE_FIRST_PAGE,High: 3/3 conditions met,Seasonality or technical issues may explain de...
3,4,content_cf56e2e2e282,75,MONITOR_REVIEW,CRITERIA_NOT_MET,Medium: 2/3 conditions met,Page is outside first-page ranking
4,5,content_7368877ea310,75,MONITOR_REVIEW,CRITERIA_NOT_MET,Medium: 2/3 conditions met,Page is outside first-page ranking
5,6,content_1bfaa38ff26c,75,MONITOR_REVIEW,CRITERIA_NOT_MET,Medium: 2/3 conditions met,Page is outside first-page ranking
6,7,content_0a91db491d14,75,MONITOR_REVIEW,CRITERIA_NOT_MET,Medium: 2/3 conditions met,Page is outside first-page ranking
7,8,content_5feee3994adb,75,MONITOR_REVIEW,CRITERIA_NOT_MET,Medium: 2/3 conditions met,Page is outside first-page ranking
8,9,content_c2d929d83eaa,75,MONITOR_REVIEW,CRITERIA_NOT_MET,Medium: 2/3 conditions met,Page is outside first-page ranking
9,10,content_b16bd7307b39,75,MONITOR_REVIEW,CRITERIA_NOT_MET,Medium: 2/3 conditions met,Page is outside first-page ranking


Top-20 review completed: 20 pages


## 4. Weak Picks and Leakage Check

### Weak Picks Audit

The five weakest selections within the top 20 are identified using the number of rule conditions satisfied and their baseline scores.

The review highlights:

- Pages with insufficient search volume.
- Pages outside the first-page ranking range.
- Pages that do not meet all priority conditions.

These pages need additional human review before an editorial action is recommended.

### Leakage Audit

The baseline score uses only these three features:

1. `days_since_last_update`
2. `impressions_90d`
3. `avg_position`

The following label-derived fields are excluded from scoring:

- `trend_direction`
- `trend_pct`
- `is_declining_label`

No product flags or future-window features are used in the scoring formula.

The decline label is used only for retrospective signal analysis, not for generating the ranking score.

### Limitations

The leakage checks verify the declared scoring feature list. They are not a complete automated audit of every possible data dependency.

The baseline is a simple heuristic, not a predictive model. Its recommendations require human validation.

In [25]:
# 1. Count conditions satisfied by each top-20 page

top20["conditions_met"] = (
    (top20["days_since_last_update"] >= 180).astype(int)
    +
    (top20["impressions_90d"] >= 500).astype(int)
    +
    (
        (top20["avg_position"] > 0) &
        (top20["avg_position"] <= 10)
    ).astype(int)
)

# 2. Find five weakest picks

weak_picks = top20.sort_values(
    ["conditions_met", "baseline_score"],
    ascending=[True, True]
).head(5)

print("FIVE WEAKEST TOP-20 PICKS")

display(
    weak_picks[
        [
            "rank",
            "content_id",
            "baseline_score",
            "conditions_met",
            "action_label",
            "what_would_make_it_wrong"
        ]
    ]
)

# 3. Leakage audit

score_features = [
    "days_since_last_update",
    "impressions_90d",
    "avg_position"
]

forbidden_features = [
    "trend_direction",
    "trend_pct",
    "is_declining_label"
]

leaked_features = set(score_features) & set(forbidden_features)

assert not leaked_features, "Label leakage detected!"

print("\nLEAKAGE AUDIT")
print("Scoring features:", score_features)
print("Label-derived features used:", list(leaked_features))

# 4. Product flag check

product_flag_columns = [
    col for col in df.columns
    if "product_flag" in col.lower()
]

used_product_flags = set(score_features) & set(product_flag_columns)

assert not used_product_flags, "Product flag leakage detected!"

print("Product flags used:", list(used_product_flags))
print("Future-window features used in scoring: None")
print("\nLeakage checks passed.")

FIVE WEAKEST TOP-20 PICKS


,rank,content_id,baseline_score,conditions_met,action_label,what_would_make_it_wrong
17,18,content_fd16e3475c29,65,2,MONITOR_REVIEW,Low search volume may limit opportunity
18,19,content_ba00ffc6318c,65,2,MONITOR_REVIEW,Low search volume may limit opportunity
19,20,content_4729b57ca036,65,2,MONITOR_REVIEW,Low search volume may limit opportunity
3,4,content_cf56e2e2e282,75,2,MONITOR_REVIEW,Page is outside first-page ranking
4,5,content_7368877ea310,75,2,MONITOR_REVIEW,Page is outside first-page ranking



LEAKAGE AUDIT
Scoring features: ['days_since_last_update', 'impressions_90d', 'avg_position']
Label-derived features used: []
Product flags used: []
Future-window features used in scoring: None

Leakage checks passed.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.